# 01 – Data Quality Audit & EDA (AdventureWorks)
Goal: prove data quality, understand the business, and write down insights you can explain in an interview.

## 0. Setup & config
**Edit only this cell** to match your tables. Defaults follow the usual AdventureWorks CSV schema
(Sales_2015/16/17, Customers, Products, Territories). If you read from your Gold layer in Databricks,
set `USE_SPARK = True` and fix `GOLD_PATH` and the table names.

In [ ]:
import os, warnings
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", None); pd.set_option("display.float_format", lambda x: f"{x:,.2f}")
sns.set_theme(style="whitegrid")

USE_SPARK  = False                 # True inside Databricks (reads Delta from the Gold layer)
DATA_DIR   = "data"                # local folder with CSV exports (used when USE_SPARK = False)
GOLD_PATH  = "/mnt/gold"           # Databricks mount point of your gold container

SALES_TABLES    = ["Sales_2015", "Sales_2016", "Sales_2017"]   # or ["fact_sales"] if you have one Gold fact table
CUSTOMERS_TABLE = "Customers"
PRODUCTS_TABLE  = "Products"
TERRITORY_TABLE = "Territories"    # optional; set to None to skip

C = dict(order_date="OrderDate", order_no="OrderNumber", product="ProductKey", customer="CustomerKey",
         territory="TerritoryKey", qty="OrderQuantity", price="ProductPrice", cost="ProductCost")

for d in ("figures", "outputs", "audit"):
    os.makedirs(d, exist_ok=True)

def load(name):
    if USE_SPARK:
        return spark.read.format("delta").load(f"{GOLD_PATH}/{name}").toPandas()   # noqa: F821 (spark exists in Databricks)
    return pd.read_csv(f"{DATA_DIR}/{name}.csv")

## 1. Load tables and build one clean order-line table

In [ ]:
sales     = pd.concat([load(t) for t in SALES_TABLES], ignore_index=True)
customers = load(CUSTOMERS_TABLE)
products  = load(PRODUCTS_TABLE)
raw_tables = {"sales": sales, "customers": customers, "products": products}

## 2. Data-quality audit (before cleaning)
Creates `audit/data_quality_report.csv` – put a screenshot of this in your README (it supports the
"monitoring / audit-ready" line of the JD).

In [ ]:
def dq_report(df, name):
    rep = pd.DataFrame({
        "table": name,
        "column": df.columns,
        "dtype": df.dtypes.astype(str).values,
        "null_count": df.isna().sum().values,
        "null_pct": (df.isna().mean() * 100).round(2).values,
        "n_unique": df.nunique().values,
    })
    rep["rows"] = len(df)
    rep["duplicate_rows"] = int(df.duplicated().sum())
    return rep

audit = pd.concat([dq_report(df, n) for n, df in raw_tables.items()], ignore_index=True)
audit.to_csv("audit/data_quality_report.csv", index=False)
audit[audit.null_count > 0].sort_values("null_pct", ascending=False).head(20)

In [ ]:
# Integrity checks on the sales fact table
checks = {
    "sales rows": len(sales),
    "duplicate sales rows": int(sales.duplicated().sum()),
    "sales with missing customer": int(sales[C["customer"]].isna().sum()),
    "sales customers not in Customers table": int((~sales[C["customer"]].dropna().isin(customers[C["customer"]])).sum()),
    "sales products not in Products table": int((~sales[C["product"]].isin(products[C["product"]])).sum()),
    "non-positive quantity": int((sales[C["qty"]] <= 0).sum()),
}
pd.Series(checks, name="value").to_frame()

## 3. Clean and build the analysis table
Every cleaning step is explained – interviewers ask *why*, not just *what*.

In [ ]:
n0 = len(sales)
df = sales.drop_duplicates().copy()                                    # exact duplicate order lines
df = df.dropna(subset=[C["customer"], C["order_date"]])                # RFM / customer analysis needs a customer + date
df[C["order_date"]] = pd.to_datetime(df[C["order_date"]])
df[C["customer"]] = df[C["customer"]].astype(int)
df = df[df[C["qty"]] > 0]

df = df.merge(products[[C["product"], C["price"], C["cost"]]], on=C["product"], how="left")
df["revenue"] = df[C["qty"]] * df[C["price"]]
df["profit"]  = df["revenue"] - df[C["qty"]] * df[C["cost"]]
df["year"], df["month"] = df[C["order_date"]].dt.year, df[C["order_date"]].dt.month
df["ym"] = df[C["order_date"]].dt.to_period("M").dt.to_timestamp()
print(f"rows before: {n0:,} | after cleaning: {len(df):,} | removed: {n0-len(df):,} ({(n0-len(df))/n0:.2%})")
df.to_csv("outputs/clean_sales.csv", index=False)

## 4. EDA – business questions

In [ ]:
# 4.1 Headline KPIs
kpi = pd.Series({
    "Orders": df[C["order_no"]].nunique(), "Customers": df[C["customer"]].nunique(),
    "Revenue": df.revenue.sum(), "Profit": df.profit.sum(), "Profit margin %": df.profit.sum()/df.revenue.sum()*100,
    "Avg order value": df.groupby(C["order_no"]).revenue.sum().mean(),
})
kpi.to_frame("value")

In [ ]:
# 4.2 Monthly revenue trend + seasonality
monthly = df.groupby("ym").agg(revenue=("revenue","sum"), orders=(C["order_no"],"nunique"))
fig, ax = plt.subplots(1, 2, figsize=(14, 4))
monthly.revenue.plot(ax=ax[0], marker="o"); ax[0].set_title("Monthly revenue"); ax[0].set_ylabel("Revenue")
df.groupby("month").revenue.sum().plot(kind="bar", ax=ax[1], color="steelblue"); ax[1].set_title("Revenue by calendar month (seasonality)")
plt.tight_layout(); plt.savefig("figures/01_revenue_trend.png", dpi=150); plt.show()

In [ ]:
# 4.3 Top products and territories
top_prod = df.groupby(C["product"]).agg(revenue=("revenue","sum"), profit=("profit","sum")).sort_values("revenue", ascending=False)
terr = df.groupby(C["territory"]).agg(revenue=("revenue","sum"), customers=(C["customer"],"nunique")).sort_values("revenue", ascending=False)
fig, ax = plt.subplots(1, 2, figsize=(14, 4))
top_prod.head(10).revenue.plot(kind="barh", ax=ax[0]).invert_yaxis(); ax[0].set_title("Top 10 products by revenue")
terr.revenue.plot(kind="bar", ax=ax[1], color="darkorange"); ax[1].set_title("Revenue by territory")
plt.tight_layout(); plt.savefig("figures/02_top_products_territory.png", dpi=150); plt.show()
share_top10 = top_prod.head(10).revenue.sum() / df.revenue.sum() * 100
print(f"Top 10 products = {share_top10:.1f}% of revenue")

In [ ]:
# 4.4 Order value distribution + outliers (IQR rule)
order_val = df.groupby(C["order_no"]).revenue.sum()
q1, q3 = order_val.quantile([.25, .75]); iqr = q3 - q1
upper = q3 + 1.5 * iqr
n_out = int((order_val > upper).sum())
print(f"Orders above IQR upper fence ({upper:,.0f}): {n_out} ({n_out/len(order_val):.1%}) – large orders, kept (they are real, not errors)")
fig, ax = plt.subplots(1, 2, figsize=(14, 4))
sns.histplot(order_val, bins=50, ax=ax[0]); ax[0].set_title("Order value distribution (right-skewed)")
sns.boxplot(x=order_val, ax=ax[1]); ax[1].set_title("Order value boxplot")
plt.tight_layout(); plt.savefig("figures/03_order_value.png", dpi=150); plt.show()

In [ ]:
# 4.5 Customer concentration (Pareto) – how much revenue do the top 20% customers bring?
cust_rev = df.groupby(C["customer"]).revenue.sum().sort_values(ascending=False)
top20 = cust_rev.head(int(len(cust_rev) * 0.2)).sum() / cust_rev.sum() * 100
print(f"Top 20% of customers generate {top20:.1f}% of revenue")
cum = cust_rev.cumsum() / cust_rev.sum() * 100
plt.figure(figsize=(7, 4)); plt.plot(np.linspace(0, 100, len(cum)), cum.values)
plt.xlabel("% of customers"); plt.ylabel("Cumulative % of revenue"); plt.title("Customer revenue concentration")
plt.savefig("figures/04_pareto.png", dpi=150); plt.show()

In [ ]:
# 4.6 Correlations
num = df.groupby(C["order_no"]).agg(items=(C["qty"],"sum"), revenue=("revenue","sum"), profit=("profit","sum"), lines=(C["product"],"nunique"))
plt.figure(figsize=(5, 4)); sns.heatmap(num.corr(), annot=True, cmap="coolwarm", fmt=".2f"); plt.title("Order-level correlations")
plt.tight_layout(); plt.savefig("figures/05_corr.png", dpi=150); plt.show()

## 5. Write your insights (do this yourself – numbers from the cells above)
1. Revenue trend / seasonality: ...
2. Top 10 products contribute __% of revenue ...
3. Top 20% customers contribute __% of revenue ...
4. Best / weakest territory: ...
5. Data quality: __ duplicate rows and __ rows with missing customer were removed because ...